In [ ]:
import numpy as np
from timeit import default_timer
import matplotlib.pyplot as plt
import numba
from Langevin_data_analyzer import Langevin_data_analyzer

In these notes we want to get a clear(er) understanding of the Langevin equation. We solve it numerically using basic 'Newton dynamics' type approach. Later on, we may use Verlet integration. We compare correlation functions to analytical results in various regimes in order to understand the concept of thermalization and how to tackle stochastic interactions.

A companion module <b>Langevin_data_analyzer</b> provides different methods for data analysis (fitting, looking for parameters, saving them, integration as well as 'comprehensive' data storage, so that every observable of one run are stored in the same object.) docstring documentation is still in progress.

# Solving the Langevin equation in and out of thermal equilibium

#### Origin of the equation

The idea of Langevin was to add a Gaussian random force, $\Gamma \eta(t)$ where $\eta(t)$ describe Gaussian fluctuation with zero mean, and $\Gamma$ quantify the amplitude of the fast moving smaller particles of the bath. The randomness is supposed to model the complex 'ultra fast' motion of the many smaller bath particle whose dynamics are much to complex to model in a deterministic way, and restore equipartition:
\begin{equation}
\nonumber
 \frac{d v}{dt} = - \frac{v}{\tau} + \frac{\Gamma}{m} \eta(t),\quad \text{where } \qquad \langle \eta(t) \rangle = 0,\qquad \langle \eta(t)\eta(t') \rangle = \delta(t-t')
\end{equation}
Taking averages over Guassian fluctuations, all terms of the form $\langle v(t) \rangle = \langle x(t) \rangle =0 $.
It can however be shown (see (https://gu-statphys.org/media/mydocs/LennartSjogren/kap6.pdf)) that (Eq 1A)
\begin{equation}
\langle v(t_1)v(t_2) \rangle =   \Bigg[v_0^2 - \frac{\tau}{2}\left(\frac{\Gamma}{m}\right)^2\Bigg] e^{-(t_1+t_2)/\tau_B} + \frac{\tau}{2}\left(\frac{\Gamma}{m}\right)^2 e^{-|t_1-t_2|/\tau_B}
\end{equation}

When considering a particle with mass $m$ moving in a medium defined by a damping coefficient $\lambda$ we get the following Newton equation and solution:
\begin{equation}
\nonumber
m\frac{d v}{dt} = -\lambda v,\quad v(t) = v_0 e^{-t/\tau},\quad \tau = m/\lambda.
\end{equation}
where $\tau$ characterises the dissipation (velocity decay) due to the bath. The stronger the damping the shorter the decaying time $\tau$, and the heavier the particle the longer it takes to decay.
Obvously, energy is not conserved as $\frac{1}{2}mv(t)^2 \to 0$ and the equipartition theorem of statistical mechanics, $\frac{1}{2}k_BT = mv^2$ is not conserved. If we want to model for example a Brownian particle moving seemingly randomly in a heat bath, and whose motion is never ending, and even to grow as one increases temperature, we expect energy to be conserved (for a fixed T). 

If we impose thermal equilibrium $\frac{1}{2}mv^2 = \frac{1}{2}k_B T$ at $t_1 = t_2 \to \infty$ then:
\begin{equation}
\text{fluctuation-dissipation}\quad\quad \langle v(t) ^2\rangle =  \frac{\tau}{2}\left(\frac{\Gamma}{m}\right)^2  \stackrel{!}{=} \frac{k_BT}{m},\quad\quad \Rightarrow\quad\quad \frac{\Gamma}{m} = \sqrt{\frac{2k_BT}{ \tau m}}
\end{equation}
Which is the <b>fluctuation-dissipation</b> theorem: Thermal equilibrium imposes that the fluctuation (of the velocity correlator) due to thermal noise (quantified by $T$) is proportional to the dissipative term $\tau$ describing loss of energy of the particle inside the bath. The bath then gives it back to the particle and so on.

## The Langevin equation

The 1d Langevin equation writes
\begin{equation}
\frac{d v}{dt} = -\frac{v}{\tau_{bath}} + v_T\sqrt{\frac{2}{\tau_{bath}}} \eta(t),\qquad \text{where }v_T=\sqrt{\frac{k_BT}{m}}, \quad\langle \eta(t) \rangle = 0,\qquad \langle \eta(t)\eta(t') \rangle = \delta(t-t')
\end{equation}
where $\eta(t)$ is a delta correlated Gaussian white noise with mean $0$ and the prefactor describes the fluctuation amplitude. As thermal equilibrium is built in by construction, dissipation, controlled by the bath correlation time $\tau_B$ is inside the fluctuation term. 

The discretized Langevin equation writes, and in units of $v_T$ ( $\tilde{x} = x/v_T$):
\begin{align}
\tilde{x}_{n+1} &= \tilde{x}_n + \tilde{v}_n dt + \frac{1}{2} F[\tilde{v}_n]dt^2\\
\tilde{v}_{n+1} &= \tilde{v}_n + F[v_n]dt\\
F[\tilde{v}_n] &= -\frac{\tilde{v}_n}{\tau} + \sqrt{\frac{2}{\tau dt }}\hat{\eta},\qquad \hat{\eta}_n \in \mathcal{N}( 0,1),\quad  \langle\hat{\eta}_n\hat{\eta}_{n'}\rangle = \delta_{n,n'} \\
\end{align}
We can retrieve the correct units by setting $  <x^2> = v_T^2 <\tilde{x}^2> $ in the end result.

the $dt$ term arises from $\lim _{t\to 0}\delta(t) = \lim _{dt\to 0}\frac{1}{dt}$ and that the full Gaussian term has the correct units. 
We can use $np.random.normal$ to generate arrays of gaussian noise, and take averages over many particle configuration (ensemble average) in order to average over the noise.


 Let us simulate $\langle x(t)x(t') \rangle = \delta(t-t')$ as an example. We simulate $N$ experiments where the particle have position $x_n(t)$, which can be respresented as a $T\times N$ matrix (N particles in T time steps):

In [ ]:
N = int(1e6)
T = int(2)
X = np.zeros((T,N))
dt= 0.001

for t in range(T):              #initialize with correct normalization
    X[t] = (1/np.sqrt(dt))*np.random.normal(0.0 ,1.0, size = N)

t1 = 0
t2 = 1
print('<X(t1)X(t2)> =', np.mean(X[t1]*X[t2]))
print('<X(t1)X(t1)> =', np.mean(X[t1]*X[t1]))

The time-lagged correlator tends to zero, while the equal time diverges as one increases the time resolution $dt$

Lets go back to business

We are mainly interested in the mean square displacement, wherre the average is unerstod as ensemble average (N identical simulations or particle 'replicas')

\begin{equation}
|\Delta x(t)|^2 := \Big\langle (x(t) -x_0)^2 \Big\rangle = \frac{1}{N}\sum^N_n [x_n(t) - x_n(0)]^2
\end{equation}

In particular, the two limiting case are the ballistic limit at $t < \tau$, when the particle has some initial inertia and has some memory from where it comes from, and the diffusive limit, where all the initial energy has been dissipated into the bath, and the particle is driven by the (diffusive) Brownian motion. Energy lost into the bath (dissipation), via the dissipative term is regained by the thermal fluctuations (fluctuation-dissipation theorem).

\begin{equation}
 |\Delta x(t)|^2 = \begin{cases}
   v_0^2 t^2 ,& \text{if } t\ll \tau \\
    2 \frac{k_B T}{m} \tau t =: 2Dt,              & t \gg \tau \text{   fluctuation=dissipation}
\end{cases}
\end{equation}

Here is the main function:

In [ ]:
@numba.njit(parallel=True)
def solve_1d_langevin_in_parallel(N_exps, N_particles, tau, v0, equilibration_time, experiment_time, dt):
    msd_ensemble            = np.zeros((N_exps, experiment_time), dtype=np.float64)              #Mean_square_displacement, multiple experiments, need to be averaged at the end
    VV0_correlator_ensemble = np.zeros((N_exps, experiment_time), dtype=np.float64)              # 'out of time' velocity correlator
    VV_temperature          = np.zeros((N_exps, experiment_time), dtype=np.float64)              # kbT = m<v^2> computes temperature from equal time correlator
    v_thermal               = 1                       #For Book-keeping correct units. we measure every velocity in units of sqrt(kbT/M). Also x is scaled as x/v_T. Therefore we need to multiply the correlator values by v_T^2
    for n_exp in numba.prange(N_exps):
        XX_correlator  = np.zeros(( experiment_time  ), dtype=np.float64)
        VV0_correlator = np.zeros(( experiment_time  ), dtype=np.float64)
        VV_correlator  = np.zeros(( experiment_time  ), dtype=np.float64)

        X  = np.zeros( N_particles, dtype=np.float64)
        V0 = np.array([ (v0*v_thermal)*np.random.normal(0,1)  for i in range(N_particles)])            #Ggenerates particle with average initial velocity v0
        X0 = X.copy()
        V  = V0.copy()                                                                                   #initiliaz the velocities
        thermal_fluctuation =   v_thermal*np.sqrt( 2/(tau*dt) )                                              #amplitu of fluctuation (Gaussian variance)
        if equilibration_time != 0:
            for _ in range(equilibration_time): 
                eta     = thermal_fluctuation * np.random.normal(0.0 ,1.0, size = N_particles)   #Ggenerates N_part gaussian values with mean 0 an variance 1
                forces  =  eta -   V / tau
                X      += V*dt  + 0.5*forces*(dt**2)
                V      += forces*dt
            X0 = X.copy()
            V0 = V.copy()
        for t in range(experiment_time):
            eta    = thermal_fluctuation * np.random.normal(0.0 ,1.0, size = N_particles)   #Ggenerates N_part gaussian values with mean 0 an variance 1
            forces =  eta -   V / tau
            X     += V*dt  + 0.5*forces*(dt**2)
            V     += forces*dt
            XX_correlator[t]  = np.mean( (X-X0)**2)  
            VV0_correlator[t] = np.mean( V*V0 )  
            VV_correlator[t]  = np.mean( V*V )

        VV_temperature[n_exp] = VV_correlator     
        msd_ensemble[n_exp]   = XX_correlator
        VV0_correlator_ensemble[n_exp] = VV0_correlator 

    return msd_ensemble, VV0_correlator_ensemble, VV_temperature

## Brownian motion: Thermal equilibrium ($v_0 = v_T$) and $t\gg \tau_B$
In thermal equilibrium, the velocity is determined by the bath temperature, hence we have: $v_0 = v_T =  \sqrt{k_BT/m}$, or in our units, $v_0 = 1$
Moreover let us look at the 'long time dynamics', when the particle has relaxed. We use a $dt = 0.01s = 10ms$ resolution and look at the behaviour of the msd at longer times. We set the bath correlation times at $0.1, 0.25, 0.5$ and $1s$ respectively. It corresponds to the time after which $v'$

We moreover equilibrate the system: We let the system evolve without taking measures for $10s$, to be sure that its velocity is close to $v_T$, and then we set our reference points $x(0)$ to be the very last configuration at the end of the equilibration period.

In [ ]:
xx_correlator_ens  = []
vv0_correlator_ens = []
heat_bath_times    = [0.1, 0.25, 0.5, 1]

In [ ]:
experiment_time    = int(5e5)
equilibration_time = 1000

N_indep_exps = 2                # 4 cpus parallelize. Parallelize is only faster than non-parallel for N_particle >> 500. Otherwise the overhead created when creating threads slows things down
N_particles  = 200
# N_particles = 20000

dt = 0.001                      #in units of seconds --> T are 10ms
v0 = 1                          #in units of sqrt(k_b T/m)

for tau in heat_bath_times:
    xx_ens, vv0_ens, _ = solve_1d_langevin_in_parallel(N_indep_exps, N_particles, tau, v0, equilibration_time, experiment_time, dt)
    xx_correlator_ens.append(xx_ens)
    vv0_correlator_ens.append(vv0_ens)
    print(f'{tau} ok!')

Let us store everything in one object and look for the parameters...

In [ ]:
brownian_data = Langevin_data_analyzer(xx_correlator_ens, vv0_correlator_ens, heat_bath_times, experiment_time, dt)

In [ ]:
brownian_data.find_diffusive_parameters(exp = 3, start = 1e4, stop = 3e4, stepsize = 500)

and plot!

In [ ]:
v_thermal = 1         #I just liked having it arround to compare with my notes, to get correct units

def msd(x,a,b):
    return b*(x**a)

color   = {'0.1': 'mediumvioletred','0.25': 'mediumpurple','0.5': 'purple','1': 'indigo'}
# counter = 0

plt.figure(figsize=(9,6))
plt.title(r'Brownian diffusion coef. from MSD: $\langle |\Delta x(t)|^2 \rangle = 2Dt$  ($v_0  =\sqrt{[k_BT/m}],t>\tau_B$)',fontsize=14)
for i, tau in enumerate(heat_bath_times):
    times = brownian_data.my_times

    msd_experiment = brownian_data.MSD_data[i]
    param_long = brownian_data.diffusive_parameters[i][0]
    ma_couleur = color[f'{tau}']


    plt.loglog(times, msd_experiment,c=f'{ma_couleur}',label=r'$\tau_B =$ {0},  $\langle |\Delta x(t)|^2 \rangle = {1} t^{{ {2} }}$, theory: $2Dt =  {3}t$  '.format(tau, round(param_long[1],2) , round(param_long[0],2) , 2*tau  ), linewidth=3)
    
    plt.loglog(times, msd(times,*param_long),c='k',linestyle=':')
    # plt.xlabel('time (ms)')
    # plt.ylabel(r' $<|\Delta r(t) |^2>$ in $m^2 /(v_T^2)$')
    plt.xlabel('Time [s]',fontsize=14)

    # plt.vlines(tau,0,1e4,color='k',linestyles='--')
    plt.legend(loc='lower right',fontsize=12)
    plt.ylabel(r' $\langle |\Delta x(t)|^2 \rangle $  ',fontsize=14)

    plt.xlim([times[1],1e2])
    plt.ylim([msd_experiment[1],1e2])
    # counter +=1
# plt.savefig('Mean_square_displacement_Brownian_Diffusion_coeff.jpg',dpi=1200)

plt.show()
 

We indeed find the expected long time behaviour, $\langle [x(t) -x(0)]^2 \rangle = 2Dt$ behaviour, as well as the expected diffusion coefficient. The Distance that the particle travels scales as $  (\sqrt{\langle  x^2\rangle }  \propto \sqrt{t}$). Let us look at velocity fluctuations. Can we  get information about dissipation from shorter time scales behaviour? The answer is yes:

## Fluctuation-dissipation

The Kubo-Green formula links the (dissipative) diffusion constant to the (fluctuating) velocity correlator at thermal equilibrium:
\begin{equation}
D = \lim_{t\to \infty} \frac{|\Delta r(t)|^2}{2t}  = \int_0^\infty dt \Big\langle v(t)v(0) \Big\rangle = v_T^2 \tau,
\end{equation}
Which can be proven by using
$ [x(t) - x(0)] = \int_0^t dt' \frac{dx(t')}{dt'} =  \int_0^t dt'v(t')$ and playing around with integration bounds and integration by parts.
We can measure the correlator and use scipy integration methods (trapezoid, simpson) to integrate. As velocity-velocity correlator is strongly fluctuating, we introduce a cut-off. Moreover, we can fit with the analytical (thermal eq) theory:
\begin{equation}
\Big\langle v(t)v(0) \Big\rangle = v_T^2 e^{-t/\tau_B}  
\end{equation}
which upon integration gives back our diffusion constant. We will use a higher resolution, and measure fluctuations on a milisecond timescale: $dt = 0.001s$ ($ms$ timescale)

In [ ]:
xx_kubo = []
vv0_kubo = []
heat_bath_times = [0.1, 0.25, 0.5, 1]

In [ ]:
T = int(1e5)
equilibration_time = 10000

N_exps = 2                   # 4 cpus parallelize. Parallelize is only faster than non-parallel for N_particle >> 500. Otherwise the overhead created when creating threads slows things down
N_particles = 20000

# tau = 0.666                      #secs
dt_kubo = 0.001                     #in units of seconds --> T are 0.1ms
v0 = 1                          #in units of sqrt(k_b T/m)

for tau in heat_bath_times:
    xx, vv, _ = solve_1d_langevin_in_parallel(N_exps,N_particles,tau,v0,equilibration_time,T, dt_kubo)
    xx_kubo.append(xx)
    vv0_kubo.append(vv)
    print(f'{tau} ok!')

In [ ]:
Kubo_experiment = Langevin_data_analyzer(T,dt_kubo,heat_bath_times,xx_kubo,vv0_kubo)

In [ ]:
Kubo_experiment.integrate_velocity_correlator(2,'simpson',3000,'timesteps')

Let us plot!

In [ ]:
 
color = {'0.1': 'mediumvioletred','0.25': 'mediumpurple','0.5': 'purple','1': 'indigo'}

plt.figure(figsize=(10,6))
plt.title(r'Fluctuation-Dissipation: $D = \int_0^{t_c} dt \langle v(t)v_0 \rangle $, Th. eq: $v_0 = \sqrt{[k_B T/m]}\equiv 1$',fontsize=15)
for i0,tau in enumerate(heat_bath_times):

    time_vec = Kubo_experiment.my_times
    my_GK = Kubo_experiment.Green_Kubo_data[i0]
    integral = Kubo_experiment.GKintegral[i0]
    t_c =  Kubo_experiment.gk_cutoff[i0]
    ma_couleur = color[f'{tau}']

    plt.semilogx(time_vec,my_GK,c=f'{ma_couleur}',label=r'D = {0}, (theory: D={1})'.format( round(integral,3), tau),linewidth=3  )
    if tau == 1:
        plt.vlines(t_c,-0.2,0.5,color=f'{ma_couleur}',linestyles=':',label='cut-off')
    else:
        plt.vlines(t_c,-0.2,0.5,color=f'{ma_couleur}',linestyles=':')
    plt.legend(loc='upper right',fontsize=14)
    plt.ylabel(r'$\langle v(t)v(0) \rangle $ in $[k_B/m]$',fontsize=15)
    plt.xlabel('Time [s]',fontsize=14)
    plt.ylim( [-0.02,1.02] )
    # Gk_integral[f'{tau}'] = integral 
# plt.savefig('Green_Kubo_Brownian_Diffusion_coeff.jpg',dpi=1200)


# Ultra short timescales $t\ll \tau_B$: Ballistic regime
Let us look at even shorter timescales. We set $dt = 0.00001s$ ($10^{-5} s$) to get a better resolution.  We expect $|\Delta x(t)|^2 = v_0^2 t^2$; irrespective of $\tau_B$. We choose $v_0$ the thermal velocity, thermalize for $1s$ ($10^5$ timesteps), and measure.

In [ ]:
xx_ballistics = []
vv0_ballistics = []
heat_bath_times = [0.1, 0.25, 0.5, 1]

T = int(1e5)
equilibration_time = int(1e5)

N_exps = 8                   # 4 cpus parallelize. Parallelize is only faster than non-parallel for N_particle >> 500. Otherwise the overhead created when creating threads slows things down
N_particles = 40000

dt_ballistics = 0.00001                     #in units of seconds --> T are 0.1ms
v0 = 1                          #in units of sqrt(k_b T/m)

for tau in heat_bath_times:
    xx, vv0, _ = solve_1d_langevin_in_parallel(N_exps,N_particles,tau,v0,equilibration_time, T,dt_ballistics)
    xx_ballistics.append(xx)
    vv0_ballistics.append(vv)
    print(f'{tau} ok!')

In [ ]:
Ballistic_experiment = Langevin_data_analyzer(T,dt_ballistics,heat_bath_times,xx_ballistics,vv0_ballistics)

In [ ]:
Ballistic_experiment.find_ballistic_parameters(2,0,2000,7)

In [ ]:
v_thermal = 1         #I just liked having it arround to compare with my notes, to get correct units

def msd(x,a,b):
    return b*(x**a)

color = {'0.1': 'mediumvioletred','0.25': 'mediumpurple','0.5': 'purple','1': 'indigo'}

# compter = 0
plt.figure(figsize=(9,6))
plt.title(r'Ballistic regime. MSD independant of $\tau_B$ (Thermal eq: $v_0  =\sqrt{[k_BT/m}] \equiv 1$)',fontsize=14)
for i, tau in enumerate(heat_bath_times):
    times = Ballistic_experiment.my_times

    msd_experiment = Ballistic_experiment.MSD_data[i]
    param_long = Ballistic_experiment.ballistic_parameters[i][0]
    ma_couleur = color[f'{tau}']


    plt.loglog(times,msd_experiment,c=f'{ma_couleur}',label=r'$\tau_B =$ {0},  $\langle |\Delta x(t)|^2 \rangle = {1} t^{{ {2} }}$, theory: $\langle |\Delta x(t)|^2 \rangle = t^2$  '.format(tau, round(param_long[1],2) , round(param_long[0],2)), linewidth=3)
    
    plt.loglog(times,msd(times,*param_long),c='k',linestyle=':')
    # plt.xlabel('time (ms)')
    # plt.ylabel(r' $<|\Delta r(t) |^2>$ in $m^2 /(v_T^2)$')
    plt.xlabel('Time [s]',fontsize=14)

    # plt.vlines(tau,0,1e4,color='k',linestyles='--')
    plt.legend(loc='upper left',fontsize=12)
    plt.ylabel(r' $\langle |\Delta x(t)|^2 \rangle $  ',fontsize=14)

    plt.xlim([times[1],times[-1]])
    plt.ylim([msd_experiment[1],1e2])
    # compter +=1
# plt.savefig('Ballistic regime.jpg',dpi=1200)

plt.show()

At these timescale the particles are not quite sensitive to the details of bath (albeit one already sees that shorter bath relaxation times give rise to some deceleration) and have the same correlation as a free particle. This regime is said to be deterministic and time reversal invariant.

## Thermalization: Out-of-Equilibrium initial conditions $v_0 < v_T$

Let us have a look at the Langevin equation outside of thermal equilibrium, ie, we start with particles NOT thermalized to the bath. The analytical solution EQ, which we saw before in the 'origin part' , is:
\begin{equation}
\Big\langle v(t)v(t) \Big\rangle =   \big[v_0^2 - v_{T_B}^2\big] e^{-2t/\tau_B} + v_T^2  
\end{equation}

Consider a heat bath with temperature $T_B$, and correlation time $\tau_B$ describing the coupling to the particle. Particles at thermal equilibrium have a thermal velocity $v_{T_B} = \sqrt{\frac{k_B T_B}{m}}$.

Let us prepare a cold particles with initial velocity $v_0 = \sqrt{\frac{k_B T_0}{m}}$, that is, they are thermalized with a temperature $T_0 < T_B$ lower than the heat bath temperature $T_B$. How fast to they 'speed up' and thermalize with the bath? We are interested in
\begin{equation}\boxed{
\frac{1}{2}k_B T(t) = \frac{1}{2} m \langle  v(t)^2\rangle }
\end{equation}
We parametrize $T_0/T_B = p_0$, where $p_0<1$ is the ratio between temperatures. The ratio of velocities read:
\begin{equation}
p_0 \equiv \frac{T_0}{T_B} ,\quad\quad \to \quad\frac{v_0}{v_{T_B}} = \sqrt{\frac{T_0}{T_B}} = \sqrt{p_0}
\end{equation}
Therefore we parametrize the initial velocities of the particles (in units of $v_{T_B}$) as $v_0 = \sqrt{p_0}$

We will set the bath relaxation time to $\tau_B= 1s$ and look for thermalization with temperature ratio $p = [0,0.125,0.25,0.5,0.75]$ and measure the instanteanous average particle temperature (in units of $T$):
\begin{equation}
p(t) = \frac{\langle  v(t)^2\rangle}{ v_T^2 } = \frac{T(t)}{T}
\end{equation}

In [ ]:
xx_thermalization = []
vv_thermalization = []
temperature_ratio = []

p = [0,0.125,0.25,0.5,0.75]
initial_velocities = np.sqrt(p)

T = int(5e5)
equilibration_time = 0

N_exps = 4                  
tau = 1
dt  = 0.001                      #in units of seconds --> T are 10ms

for v0 in initial_velocities:
    xx, vv, T_ratio = solve_1d_langevin_in_parallel(N_exps,N_particles,tau,dt,v0,equilibration_time,T)
    temperature_ratio.append(np.mean(T_ratio, axis=0))
    xx_thermalization.append(xx)
    vv_thermalization.append(vv)
    print(f'{v0} ok!')

In [ ]:
i = 0
time_array = np.array([i for i in range(T)  ])
color = {'0': 'mediumvioletred','1': 'mediumpurple','2': 'purple','3': 'indigo', '4': 'midnightblue', }
plt.figure(figsize=(12,6))
plt.title(r'Thermalization of a cold particle in a heat bath in 1d : $k_B T(t) =m\langle v(t)^2 \rangle $ ',fontsize=14)

ma_couleur = color[f'{0}']
plt.loglog(time_array,temperature_ratio[0],c=ma_couleur,label=r'$T_0 =   {0}T_B   $'.format(p[0]),linewidth=3)
for i in range(1,5):
    ma_couleur = color[f'{i}']
    plt.loglog(time_array,temperature_ratio[i],c=ma_couleur,label=r' $T_0 =   {0}  T_B  $'.format(p[i]),linewidth=3)
plt.xlabel('Time [ms]',fontsize=14)

plt.vlines(tau/dt,0,1e4,color='k',linestyles='--',label=r'Bath corr. time $\tau_B$')
plt.hlines(tau,time_array[1],time_array[-1],color='k',linestyles=':')
plt.xlim([time_array[1],int(1e4)])
plt.ylim([temperature_ratio[0][1],1.1])

plt.legend(loc='lower right',fontsize=12)
plt.ylabel(r' $T(t)/T_B $  ',fontsize=14)

# plt.savefig('Thermalization of a cold particle in a heat bath in 1d.jpg',dpi=1200)
